# Part 2: Model Training & Strategy Formulation
In this notebook, we will use the engineered features from Part 1 to train a predictive model.

**Prediction Target:** Regression. We will predict the 5-day forward return.
**Model:** XGBoost Regressor.


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import os
import glob

# ==========================================
# IMPORTANT: UPDATE THIS PATH FOR GOOGLE COLAB
# ==========================================
# If running in Colab, mount drive first:
# from google.colab import drive
# drive.mount('/content/drive')
# data_dir = '/content/drive/MyDrive/PreCog/processed_data'

# Defaulting to local path for now:
data_dir = 'processed_data' 


## 1. Data Loading and Target Definition
We load the processed CSVs, and define our target variable. Since we want to predict the 5-day forward return, we shift the `Ret_5d` column backwards by 5 days.

In [ ]:
all_files = glob.glob(os.path.join(data_dir, "*.csv"))
df_list = []

for file in all_files:
    df = pd.read_csv(file)
    df['Asset'] = os.path.basename(file).replace('.csv', '')
    
    # Define Target: Shift Ret_5d backwards by 5 days so today's row contains the return 5 days from now
    df['Target_5d'] = df['Ret_5d'].shift(-5)
    
    df_list.append(df)

full_df = pd.concat(df_list, ignore_index=True)

# Drop rows where target is NaN (the last 5 days of every asset)
# Also drop rows where features are NaN (first 50 days due to SMA_50)
full_df.dropna(inplace=True)

# Sort chronologically
full_df['Date'] = pd.to_datetime(full_df['Date'])
full_df.sort_values(by=['Date', 'Asset'], inplace=True)
print(f"Total dataset size after dropping NaNs: {full_df.shape}")


## 2. Chronological Train/Test Split
In time-series finance, we must **never** randomly split data, or we will introduce look-ahead bias. We will train on the first 80% of dates, and test on the remaining 20%.

In [ ]:
dates = full_df['Date'].unique()
split_idx = int(len(dates) * 0.8)
train_dates = dates[:split_idx]
test_dates = dates[split_idx:]

train_df = full_df[full_df['Date'].isin(train_dates)].copy()
test_df = full_df[full_df['Date'].isin(test_dates)].copy()

features = [
    'Ret_1d', 'Ret_5d', 'RSI_14', 'MACD', 'MACD_Signal', 'MACD_Hist',
    'SMA_10', 'SMA_50', 'Dist_SMA_50', 'BB_Upper', 'BB_Lower',
    'Vol_20d', 'Vol_SMA_20', 'Vol_ROC'
]
target = 'Target_5d'

X_train, y_train = train_df[features], train_df[target]
X_test, y_test = test_df[features], test_df[target]

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")


## 3. XGBoost Regression
We train an XGBRegressor to predict the exact 5-day forward return.

In [ ]:
model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    tree_method='hist' # Uses GPU if available in Colab
)

model.fit(X_train, y_train)

# Evaluate Baseline
preds = model.predict(X_test)
mse = mean_squared_error(y_test, preds)
r2 = r2_score(y_test, preds)
print(f"Test MSE: {mse:.6f}")
print(f"Test R^2: {r2:.6f}")


## 4. Feature Importance
Let's see which of the 14 features we engineered in Part 1 were the most useful for the model.

In [ ]:
importances = model.feature_importances_
feat_imp = pd.Series(importances, index=features).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
feat_imp.plot(kind='bar')
plt.title('XGBoost Feature Importance')
plt.ylabel('Relative Importance')
plt.show()


## 5. Strategy Logic: Translating Predictions to Signals
Now that we have predictions for the forward return, we need to translate them into actionable trading signals (-1, 0, 1) for the backtester.

**Logic:**
Since it's regression, we will generate signals based on the predicted magnitude. If the predicted return is exceptionally high, we buy. If it's exceptionally low, we short.
* Top 20% predicted returns for the day -> Go Long (+1)
* Bottom 20% predicted returns for the day -> Go Short (-1)
* Middle 60% -> Do Nothing (0)


In [ ]:
# Generate predictions for the entire dataset (for backtesting)
full_df['Predicted_Ret_5d'] = model.predict(full_df[features])

def generate_signals(group):
    # Calculate quantiles for each day
    upper_thresh = group['Predicted_Ret_5d'].quantile(0.80)
    lower_thresh = group['Predicted_Ret_5d'].quantile(0.20)
    
    conditions = [
        (group['Predicted_Ret_5d'] > upper_thresh),
        (group['Predicted_Ret_5d'] < lower_thresh)
    ]
    choices = [1, -1]
    
    group['Signal'] = np.select(conditions, choices, default=0)
    return group

# Apply logic day-by-day
print("Generating signals...")
full_df = full_df.groupby('Date').apply(generate_signals)

# Check signal distribution
print(full_df['Signal'].value_counts(normalize=True))

# Save the final signals to be used in Part 3
if not os.path.exists('signals'):
    os.makedirs('signals')
full_df[['Date', 'Asset', 'Close', 'Target_5d', 'Predicted_Ret_5d', 'Signal']].to_csv('signals/strategy_signals.csv', index=False)
print("Signals saved to signals/strategy_signals.csv")
